# Clase 11: Data warehouses, data marts, data lakes, ETL e integración

**Objetivos de aprendizaje:**
- Diferenciar warehouse, mart y lake.
- Describir los tres pasos de ETL.
- Construir un mini-pipeline ETL con pandas y sqlite3.
- Explicar qué es la integración de datos y los metadatos.

**Duración estimada:** 110 min


## Tres formas de centralizar datos

Imaginad una empresa con datos repartidos en veinte sistemas distintos. Si cada área analiza sus datos por separado, tarde o temprano dos informes se van a contradecir. Por eso existen estos tres conceptos:

- **Data warehouse**: repositorio central, ya modelado y listo para el análisis — la "fuente única de verdad".
- **Data mart**: una porción de ese warehouse para un área de negocio concreta (ventas, finanzas...), con seguridad y rendimiento aislados.
- **Data lake**: almacenamiento en formato nativo (crudo) de cualquier tipo de dato, etiquetado con metadatos, sin excluir nada — útil para analítica avanzada o como zona de preparación (staging) antes de un warehouse.

## ETL: Extract, Transform, Load

El proceso que lleva los datos desde su origen hasta el warehouse tiene tres pasos:

1. **Extract**: sacar los datos de su fuente (por lotes, con herramientas como Stitch o Blendo; o en streaming, con Kafka, Storm o Samza).
2. **Transform**: estandarizar formatos, eliminar duplicados, filtrar, enriquecer, dividir campos, aplicar reglas de negocio.
3. **Load**: cargar el resultado en su destino (carga inicial, incremental o refresco completo), verificando que no haya nulos inesperados ni fallos silenciosos.

Vamos a construir los tres pasos, en vivo, con datos que imitan los problemas típicos de datos reales.

In [ ]:
import pandas as pd
import numpy as np
import sqlite3

# EXTRACT: datos "crudos" simulados, con problemas típicos —
# una fila duplicada, un cliente sin nombre, un importe vacío,
# mayúsculas/minúsculas inconsistentes.
datos_crudos = pd.DataFrame({
    "id_venta":  [1, 2, 3, 4, 4, 5],
    "cliente":   ["Ana Torres", "luis peña", "CARLA RUIZ", "Diego Soto", "Diego Soto", None],
    "producto":  ["Camiseta", "camiseta", "Auriculares", None, None, "Mochila"],
    "importe":   [15.99, 15.99, 39.90, 25.00, 25.00, np.nan],
})

print("Datos crudos:")
print(datos_crudos)

In [ ]:
# TRANSFORM: limpiamos paso a paso
datos_limpios = datos_crudos.drop_duplicates()
datos_limpios = datos_limpios.dropna(subset=["cliente"])
datos_limpios["importe"] = datos_limpios["importe"].fillna(0)
datos_limpios["cliente"] = datos_limpios["cliente"].str.title()
datos_limpios["producto"] = datos_limpios["producto"].fillna("Sin especificar").str.title()
datos_limpios["importe"] = pd.to_numeric(datos_limpios["importe"], errors="coerce").fillna(0)
datos_limpios = datos_limpios[datos_limpios["importe"] >= 0]

print("Datos transformados:")
print(datos_limpios)

In [ ]:
# LOAD: cargamos el resultado limpio a una tabla relacional,
# nuestro mini-warehouse en memoria.
conexion = sqlite3.connect(':memory:')
datos_limpios.to_sql("ventas", conexion, index=False, if_exists="replace")

verificacion = pd.read_sql_query("SELECT * FROM ventas", conexion)
print("Datos cargados en la tabla 'ventas':")
print(verificacion)

conexion.close()

## Más allá de ETL: pipelines, integración y metadatos

- Un **pipeline de datos** es más amplio que ETL: incluye batch y streaming, y no siempre termina en un warehouse (herramientas: Apache Beam, DataFlow). Todo ETL es un pipeline, pero no todo pipeline es un ETL.
- La **integración de datos** (según Gartner) son "las prácticas, técnicas arquitectónicas y herramientas que permiten a las organizaciones ingerir, transformar, combinar y proveer datos a través de distintos tipos de datos" — un concepto todavía más amplio, que incluye consistencia entre aplicaciones, gestión de datos maestros y migraciones. Herramientas reales: IBM Information Server, Talend, Informatica, entre otras.
- Los **metadatos** ("datos sobre los datos") son de tres tipos: **técnicos** (estructura: tablas, columnas), **de proceso** (cómo se movieron los datos: tiempos, origen/destino) y **de negocio** (qué significan los datos para la organización). Se organizan en un **catálogo de datos**, clave para la gobernanza y la procedencia de datos (trazar un error hasta su causa raíz).

## Resumen de la clase

- Warehouse = centralizado y modelado; mart = porción por área de negocio; lake = crudo, sin transformar.
- ETL = Extract + Transform + Load; un pipeline de datos es más amplio; la integración de datos es más amplia todavía.
- `pandas` (`drop_duplicates`, `dropna`, `fillna`, normalización de texto) + `sqlite3` (`to_sql`) permiten construir un ETL real, aunque pequeño.
- Los metadatos técnicos, de proceso y de negocio viven en un catálogo de datos, base de la gobernanza y la procedencia de datos.

**Capítulo del manual:** [13 — Warehouses, marts, lakes, ETL y pipelines](../manual/13_warehouses_etl_pipelines.md)

**Notebook de práctica:** [clase11_practica_warehouses_etl_pipelines.ipynb](../notebooks_practica/clase11_practica_warehouses_etl_pipelines.ipynb)